# Concepts Notebook — Probability, Statistics & Financial Products

Every core concept the strategy uses, as a small function example

In [ ]:
import numpy as np, pandas as pd
rng = np.random.default_rng(0)

## A. Financial Product Concepts

### A1. A futures calendar spread
The spread is the price of one delivery month minus another of the *same* commodity:
`spread = P(front) - P(deferred)`. Trading it isolates curve **shape**: a parallel
shift in all contracts (a macro/weather shock hitting every month) largely cancels.

In [ ]:
def calendar_spread(front, deferred):
    return front - deferred

# example: front $3.20, Jan contract $3.55
print("spread =", calendar_spread(3.20, 3.55), "$/MMBtu  (front is cheaper -> contango)")

### A2. Contango vs backwardation
* **Contango**: deferred > front (spread < 0). Typical when storage is comfortable.
* **Backwardation**: front > deferred (spread > 0). Typical when near-term gas is scarce.
Storage stress pushes the curve between these states — that is the edge.

In [ ]:
def curve_state(spread):
    return "backwardation" if spread > 0 else ("contango" if spread < 0 else "flat")
for s in (+0.15, -0.35, 0.0):
    print(f"spread {s:+.2f} -> {curve_state(s)}")

### A3. Inverse-volatility position sizing
Risk a fixed dollar amount per unit of spread volatility. Quieter spread → bigger
position; noisier spread → smaller. Keeps risk-per-trade roughly constant.

In [ ]:
def size_contracts(capital, risk_per_sd, spread_daily_sd, point_value=10_000):
    # point_value = $ per 1.00 spread move per contract (NG = 10,000 MMBtu)
    return (capital * risk_per_sd) / (spread_daily_sd * point_value)

print(round(size_contracts(100_000, 0.005, 0.04), 2), "contracts  (quiet spread)")
print(round(size_contracts(100_000, 0.005, 0.12), 2), "contracts  (volatile spread)")

## B. Statistics Concepts

### B1. Z-score (standardize a signal)
How many standard deviations a value sits from its rolling mean. The trade trigger.

In [ ]:
def zscore_last(series, window):
    s = pd.Series(series)
    mu = s.rolling(window).mean().iloc[-1]
    sd = s.rolling(window).std().iloc[-1]
    return (s.iloc[-1] - mu) / sd

x = rng.normal(0, 1, 200); x[-1] = 3.0
print("z of last point:", round(zscore_last(x, 156), 2))

### B2. Rolling realized volatility
Standard deviation of recent changes — used both for sizing and the regime gate.

In [ ]:
def realized_vol(changes, window):
    return pd.Series(changes).rolling(window).std().iloc[-1]
print("13-pt realized vol:", round(realized_vol(rng.normal(0,0.05,60), 13), 4))

### B3. OLS regression & residualization (the analytical core)
Regress the raw signal on basis changes; the **residual** is the part of stress
*not* explained by what basis traders already trade. We trade the residual.

In [ ]:
def residualize(y, X):
    import numpy as np
    X1 = np.column_stack([np.ones(len(X)), X])
    beta, *_ = np.linalg.lstsq(X1, y, rcond=None)
    fitted = X1 @ beta
    return y - fitted, beta

n=200
basis = rng.normal(0,1,n)
true_signal = rng.normal(0,1,n)
y = 0.7*basis + true_signal           # raw signal contaminated by basis
resid, beta = residualize(y, basis.reshape(-1,1))
print("recovered basis loading ~0.7:", round(beta[1],2))
print("corr(resid, basis) ~0:", round(float(np.corrcoef(resid, basis)[0,1]), 3))

### B4. Sharpe ratio + bootstrap confidence interval
The proposal judges performance on **ranges**, not point estimates, because weekly
data is autocorrelated and samples are few. Bootstrap resamples returns to get a CI.

In [ ]:
def sharpe(returns, ppy=52):
    r = np.asarray(returns); 
    return 0.0 if r.std()==0 else np.sqrt(ppy)*r.mean()/r.std()

def bootstrap_sharpe_ci(returns, n_boot=2000, ci=0.90, seed=1):
    r = np.asarray(returns); g = np.random.default_rng(seed)
    stats = [sharpe(g.choice(r, len(r), replace=True)) for _ in range(n_boot)]
    lo, hi = np.percentile(stats, [(1-ci)/2*100, (1+ci)/2*100])
    return sharpe(r), (round(lo,2), round(hi,2))

rets = rng.normal(0.001, 0.02, 260)
pt, ci = bootstrap_sharpe_ci(rets)
print("Sharpe", round(pt,2), "90% CI", ci, "-> CI excludes 0?", ci[0] > 0)

### B5. Paired Sharpe test (does DSI beat Agent One?)
The deploy decision (section 7): DSI must beat Agent One by ≥0.4 Sharpe with a 90%
bootstrap CI on the **difference** that excludes zero. Resample the two return series
*together* (paired) to preserve their correlation.

In [ ]:
def paired_sharpe_diff_ci(ret_a, ret_b, n_boot=2000, ci=0.90, seed=2):
    a, b = np.asarray(ret_a), np.asarray(ret_b); g = np.random.default_rng(seed)
    diffs=[]
    for _ in range(n_boot):
        idx = g.integers(0, len(a), len(a))   # same index for both = paired
        diffs.append(sharpe(a[idx]) - sharpe(b[idx]))
    lo, hi = np.percentile(diffs, [(1-ci)/2*100, (1+ci)/2*100])
    return round(sharpe(a)-sharpe(b),2), (round(lo,2), round(hi,2))

dsi_ret = rng.normal(0.0015, 0.02, 260)
one_ret = rng.normal(0.0005, 0.02, 260)
d, ci = paired_sharpe_diff_ci(dsi_ret, one_ret)
print("Sharpe(DSI)-Sharpe(One) =", d, " 90% CI", ci,
      "-> deploy DSI?", (d >= 0.4 and ci[0] > 0))

## C. Probability concepts

### C1. Hit rate & expectancy
Hit rate = fraction of winning trades. Expectancy = avg win × p(win) − avg loss × p(loss).
The proposal targets a 55–62% hit rate.

In [ ]:
def hit_rate(trade_pnls):
    t = np.asarray(trade_pnls); return float((t > 0).mean())
def expectancy(trade_pnls):
    t = np.asarray(trade_pnls); wins=t[t>0]; losses=t[t<=0]
    pw = len(wins)/len(t)
    return pw*(wins.mean() if len(wins) else 0) + (1-pw)*(losses.mean() if len(losses) else 0)
pnls=[120,-60,80,-40,200,-90,50]
print("hit rate", round(hit_rate(pnls),2), "| expectancy/trade $", round(expectancy(pnls),1))

### C2. Value at Risk (portfolio limit)
1-day 99% VaR from the spread covariance matrix — the proposal's 2%-of-equity book limit.

In [ ]:
from scipy.stats import norm
def parametric_var(weights, cov, equity, conf=0.99):
    w = np.asarray(weights)
    port_sd = np.sqrt(w @ np.asarray(cov) @ w)
    return norm.ppf(conf) * port_sd * equity

cov = np.array([[0.0004,0.0001],[0.0001,0.0003]])  # 2 spreads' daily return cov
var = parametric_var([0.6,0.4], cov, 100_000)
print("1-day 99% VaR: $", round(var,0), " -> within 2% limit?", var <= 0.02*100_000)

### C3. Why the convex stress transform is non-linear
A system at 95% utilization absorbs a shock far worse than one at 70%. The transform
`g(u)=sign(u)(e^{k|u|}-1)/(e^{k}-1)` is ~flat in the middle and steep near the limit,
encoding "binding constraint" behaviour.

In [ ]:
def convex_stress(u, k=4.0):
    u = np.clip(u, -1.2, 1.2)
    return np.sign(u) * (np.expm1(k*np.abs(u)) / np.expm1(k))
for u in (0.3, 0.7, 0.9, 0.95):
    print(f"utilization {u:.2f} -> stress {convex_stress(u):.3f}")